# An Audio Research Notebook on Venice

Add sources, ask questions that cite them, then generate a two-host audio overview and play it back here.

This notebook accompanies [Building an Audio Research Notebook](https://docs.venice.ai/guides/projects/audio-research-notebook), which explains the reasoning behind each step. Run the cells in order.

You need a Venice API key from [venice.ai/settings/api](https://docs.venice.ai/guides/getting-started/generating-api-key). A full run scrapes three pages, embeds them, makes two chat completions, and synthesizes about six minutes of speech, so it consumes a small amount of credit.

## Setup

Store the key with the key icon in the Colab sidebar, as a secret named `VENICE_API_KEY`, so it is not saved into the notebook when you share it. If no secret is set you will be prompted for it, and the value stays in memory.

Run this cell first. The configuration cell below reads the key as it is imported.

In [ ]:
%pip install -q requests

import os


def load_api_key() -> str:
    try:
        from google.colab import userdata

        return userdata.get('VENICE_API_KEY')
    except Exception:
        pass
    if os.environ.get('VENICE_API_KEY'):
        return os.environ['VENICE_API_KEY']
    from getpass import getpass

    return getpass('Venice API key: ')


os.environ['VENICE_API_KEY'] = load_api_key()
print('Key loaded.')

## Configuration

`HOSTS` maps a host name to a voice. Both voices belong to `tts-xai-v1`, and that matters: voices belong to models, and sending a voice from one family to a model from another is the most common first mistake with the speech endpoint.

`sources` and `chunks` are the entire state of the notebook.

In [ ]:
import io
import json
import os
import re
import wave
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

BASE_URL = "https://api.venice.ai/api/v1"
HEADERS = {"Authorization": f"Bearer {os.environ['VENICE_API_KEY']}"}

EMBED_MODEL = "text-embedding-bge-m3"
TTS_MODEL = "tts-xai-v1"
HOSTS = {"Ana": "luna", "Marco": "orion"}

sources = []
chunks = []

## Pick the current model

Hardcoding a chat model guarantees the project ages. `/models/traits` reports which model currently holds each role, so this asks for the current default instead of naming one.

In [ ]:
def default_text_model():
    response = requests.get(
        f"{BASE_URL}/models/traits", headers=HEADERS, params={"type": "text"}, timeout=60
    )
    response.raise_for_status()
    return response.json()["data"]["default"]


CHAT_MODEL = default_text_model()


def chat(messages, **options):
    response = requests.post(
        f"{BASE_URL}/chat/completions",
        headers=HEADERS,
        json={"model": CHAT_MODEL, "messages": messages, **options},
        timeout=300,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
print('Using', CHAT_MODEL)

## Add sources

A source is a URL or a file on disk, and Venice has an endpoint for each. Both return plain text, so nothing downstream cares which one you used.

In [ ]:
def read_url(url):
    response = requests.post(
        f"{BASE_URL}/augment/scrape", headers=HEADERS, json={"url": url}, timeout=180
    )
    response.raise_for_status()
    return response.json()["content"]


def read_file(path):
    with open(path, "rb") as handle:
        response = requests.post(
            f"{BASE_URL}/augment/text-parser",
            headers=HEADERS,
            files={"file": (Path(path).name, handle)},
            timeout=180,
        )
    response.raise_for_status()
    return response.json()["text"]

## Chunk and embed

Embedding a whole document produces one vector that averages everything it says, which is too blunt to retrieve a specific claim. Splitting on paragraph boundaries produces vectors that each mean something.

In [ ]:
def split(text, limit=1200):
    """Pack paragraphs into chunks without cutting one in half."""
    packed, current = [], ""
    for para in re.split(r"\n\s*\n", text):
        para = para.strip()
        if not para:
            continue
        if current and len(current) + len(para) + 2 > limit:
            packed.append(current)
            current = para
        else:
            current = f"{current}\n\n{para}" if current else para
    if current:
        packed.append(current)
    return packed


def embed(texts):
    vectors = []
    for start in range(0, len(texts), 64):
        response = requests.post(
            f"{BASE_URL}/embeddings",
            headers=HEADERS,
            json={"model": EMBED_MODEL, "input": texts[start : start + 64]},
            timeout=180,
        )
        response.raise_for_status()
        vectors.extend(row["embedding"] for row in response.json()["data"])
    return vectors

In [ ]:
def add_source(title, ref):
    text = read_url(ref) if ref.startswith("http") else read_file(ref)
    number = len(sources) + 1
    sources.append({"number": number, "title": title, "ref": ref})

    pieces = split(text)
    for piece, vector in zip(pieces, embed(pieces)):
        magnitude = sum(x * x for x in vector) ** 0.5
        chunks.append(
            {"source": number, "title": title, "text": piece,
             "vector": vector, "magnitude": magnitude}
        )
    print(f"[{number}] {title}: {len(text)} characters, {len(pieces)} chunks")

Now add some sources. These three Venice pages cover overlapping ground, which makes the citations in the next section more interesting. Swap in your own URLs.

In [ ]:
add_source('Venice Privacy', 'https://docs.venice.ai/overview/privacy')
add_source('TEE and E2EE Models', 'https://docs.venice.ai/guides/features/tee-e2ee-models')
add_source('VVV and DIEM', 'https://docs.venice.ai/overview/vvv-diem')

print(f'{len(chunks)} chunks from {len(sources)} sources')

### Optional: add a PDF from your machine

This cell waits for you to choose a file, so skip it if you only want web sources. The text parser accepts PDF, Word, Excel, and plain text up to 25 MB.

In [ ]:
try:
    from google.colab import files

    for name in files.upload():
        add_source(name, name)
except ImportError:
    print('Not running in Colab, skipping the upload.')

## Ask a question

Two instructions do the work of grounding: answer only from the notes, and say so when the notes fall short. Without the second one a model quietly fills the gap from memory, which is the failure mode you are designing out.

Numbering the notes gives the model a citation vocabulary, and parsing the brackets back out tells you which sources actually carried the answer.

In [ ]:
def retrieve(question, k=6):
    query = embed([question])[0]
    query_magnitude = sum(x * x for x in query) ** 0.5

    def similarity(chunk):
        dot = sum(a * b for a, b in zip(query, chunk["vector"]))
        return dot / (query_magnitude * chunk["magnitude"])

    return sorted(chunks, key=similarity, reverse=True)[:k]

In [ ]:
def ask(question, k=6):
    hits = retrieve(question, k)
    notes = "\n\n".join(f"[{h['source']}] {h['title']}\n{h['text']}" for h in hits)
    answer = chat(
        [
            {"role": "system", "content": (
                "Answer only from the numbered notes. Cite every claim with the bracket number "
                "of the note it came from. If the notes do not answer the question, say so "
                "instead of filling the gap.")},
            {"role": "user", "content": f"Notes:\n\n{notes}\n\nQuestion: {question}"},
        ],
        temperature=0.2,
    )
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
    return answer, [s for s in sources if s["number"] in cited]

In [ ]:
from IPython.display import Markdown, display

answer, cited = ask('How does Venice keep my prompts private, and what do I give up?')

display(Markdown(answer))
print('Sources:', ', '.join(f"[{s['number']}] {s['title']}" for s in cited))

## Write the overview script

A summary is something you read; an overview is something you listen to. Dialogue works better in audio because the turn-taking does the pacing, and a question from one host introduces the next idea naturally.

Asking for JSON with a schema is what makes the result renderable: the `enum` on `speaker` guarantees every turn maps to a voice you have.

In [ ]:
DIALOGUE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "dialogue",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["turns"],
            "properties": {
                "turns": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["speaker", "text"],
                        "properties": {
                            "speaker": {"type": "string", "enum": list(HOSTS)},
                            "text": {"type": "string"},
                        },
                    },
                }
            },
        },
    },
}


def write_script(turns=16):
    """Ask a chat model for a two-host dialogue grounded in the sources."""
    spread = chunks[:: max(1, len(chunks) // 12)][:12]
    notes = "\n\n".join(f"{c['title']}\n{c['text']}" for c in spread)
    hosts = " and ".join(HOSTS)
    raw = chat(
        [
            {"role": "system", "content": (
                f"You write podcast dialogue for two hosts, {hosts}. Ground every statement in "
                "the supplied notes. Write for the ear: no markdown, no URLs, no bracket "
                "citations, no stage directions. Spell out abbreviations the first time they "
                "appear. Vary the length of turns. Open with a hook and close with a takeaway.")},
            {"role": "user", "content": f"Notes:\n\n{notes}\n\nWrite about {turns} turns."},
        ],
        temperature=0.7,
        response_format=DIALOGUE_SCHEMA,
    )
    return json.loads(raw)["turns"]

In [ ]:
turns = write_script(16)

print(f'{len(turns)} turns\n')
for turn in turns[:4]:
    print(f"{turn['speaker']}: {turn['text']}\n")

## Render it

Each turn is one speech request, with the voice chosen by who is speaking. Reading the frames out of each clip rather than saving files and stitching them afterwards is what keeps the join clean, because concatenating encoded audio such as MP3 does not work reliably.

The output header comes from the first clip rather than from constants, so the sample rate is right for whichever model you chose, and a quarter second of silence between turns gives the ear a beat to register that the speaker changed.

Rendering six minutes of speech takes somewhere between half a minute and three minutes.

In [ ]:
def speak(turn):
    response = requests.post(
        f"{BASE_URL}/audio/speech",
        headers=HEADERS,
        json={"model": TTS_MODEL, "voice": HOSTS[turn["speaker"]],
              "input": turn["text"], "response_format": "wav"},
        timeout=300,
    )
    response.raise_for_status()
    with wave.open(io.BytesIO(response.content)) as clip:
        return clip.getparams(), clip.readframes(clip.getnframes())

In [ ]:
def audio_overview(turns, path="overview.wav", pause_seconds=0.25):
    with ThreadPoolExecutor(max_workers=4) as pool:
        rendered = list(pool.map(speak, turns))

    params = rendered[0][0]
    silence = b"\x00" * int(params.framerate * params.sampwidth * params.nchannels * pause_seconds)
    with wave.open(path, "wb") as out:
        out.setnchannels(params.nchannels)
        out.setsampwidth(params.sampwidth)
        out.setframerate(params.framerate)
        for position, (_, frames) in enumerate(rendered):
            if position:
                out.writeframes(silence)
            out.writeframes(frames)
    return path

In [ ]:
from IPython.display import Audio

audio_overview(turns, 'overview.wav')

Audio('overview.wav')

## Next steps

- [Building a Private RAG Bot](https://docs.venice.ai/guides/projects/private-rag-bot), the same retrieval pipeline with a real vector database and re-ranking
- [Cited Answers with Web Search](https://docs.venice.ai/guides/tools/cited-web-answers), find the sources automatically instead of naming them
- [Voice Cloning](https://docs.venice.ai/guides/media/voice-cloning), host the overview in your own voice
- [Document Processing](https://docs.venice.ai/guides/tools/document-processing), everything the text parser accepts and what it returns